In [3]:
import pandas as pd
import unidecode
import re

# Extract a list of postcodes, nis codes (at municipality, province, and region levels), and names associated to the nis codes

## 1. Postcodes

In [4]:
pc = pd.read_excel('source_data/Conversion Postal code_Refnis code_va01012019.xlsx')
postcodes = list(pc['Postal code'].unique())
print(len(postcodes))

1149


## 2. NIS codes

In [5]:
levels = {
    'Region': 1,
    'Province': 2,
    'Arrondissement': 3,
    'Commune': 4,
}

nis = pd.read_excel('source_data/TU_COM_REFNIS.xlsx', parse_dates=['DT_VLDT_START', 'DT_VLDT_END'])
nis.loc[nis['DT_VLDT_END'] == '31/12/9999', 'DT_VLDT_END'] = pd.NaT

existing_nis = {}
valid_nis = {}

for level, level_id in levels.items():
    existing_nis[level] = sorted(list(nis.loc[nis['LVL_REFNIS'] == level_id, 'CD_REFNIS'].unique()))

    valid_nis[level] = sorted(list(nis[(nis['LVL_REFNIS'] == level_id) & (nis['DT_VLDT_END'].isnull())]['CD_REFNIS'].unique()))
    print(f'{level}: {len(valid_nis[level])} valid NIS codes, {len(existing_nis[level])} existing NIS codes')

Region: 3 valid NIS codes, 3 existing NIS codes
Province: 10 valid NIS codes, 11 existing NIS codes
Arrondissement: 43 valid NIS codes, 44 existing NIS codes
Commune: 581 valid NIS codes, 2702 existing NIS codes


/home/robin/.virtualenvs/explore/lib/python3.8/site-packages/pandas/core/tools/datetimes.py:1063: UserWarning: Parsing '31/12/9999' in DD/MM/YYYY format. Provide format or specify infer_datetime_format=True for consistent parsing.
  cache_array = _maybe_cache(arg, format, cache, convert_listlike)


## 3. Noms

In [6]:
def normalize_string(s):
    s = s.strip()  # Remove leading and trailing whitespace
    s = s.lower()  # Convert to lowercase
    s = unidecode.unidecode(s)  # Remove accents
    s = s.replace('-', ' ')  # Replace dashes with spaces
    s = re.sub(r'\s+', ' ', s)  # Replace multiple spaces with a single space
    
    return s

existing_names = {}
normalised_names = {}

for level, level_id in levels.items():
    # Allemand
    existing_names[level] = list(nis.loc[nis['LVL_REFNIS'] == level_id, 'TX_REFNIS_DE'].unique())
    # Français
    existing_names[level] += list(nis.loc[nis['LVL_REFNIS'] == level_id, 'TX_REFNIS_FR'].unique())
    # Néerlandais
    existing_names[level] += list(nis.loc[nis['LVL_REFNIS'] == level_id, 'TX_REFNIS_NL'].unique())

    if level == 'Commune':
        existing_names[level] += list(pc['Nom commune'].unique())
        existing_names[level] += list(pc['Gemeentenaam'].unique())

    existing_names[level] = sorted(list(set(existing_names[level])))
    normalised_names[level] = sorted(list(set([normalize_string(name) for name in existing_names[level]])))

    print(f'{level}: {len(normalised_names[level])} normalised names, {len(existing_names[level])} existing names')

    

Region: 9 normalised names, 9 existing names
Province: 33 normalised names, 33 existing names
Arrondissement: 135 normalised names, 135 existing names
Commune: 2956 normalised names, 2964 existing names


## Save everything in a js file

In [7]:
possible_values = {
    'postcodes': sorted(postcodes),
    'nis': existing_nis,
    'normalised_names': normalised_names,
    'names': existing_names
}

import json
import numpy as np

def default(o):
    if isinstance(o, np.int64): 
        return int(o)  
    raise TypeError

with open('../geodata/possible_geo_identifiers.js', 'w') as f:
    f.write('const possible_geo_identifiers = ')
    json.dump(possible_values, f, default=default, indent=2)
    f.write(';')